<a href="https://colab.research.google.com/github/AnshulKumar79/SwasthyaSahay_vision/blob/main/CNN_model_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import json


kaggle_credentials = {
    "username": "USERNAME",
    "key": "KAGGLE_KEY"
}

os.makedirs('/root/.kaggle', exist_ok=True)

with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump(kaggle_credentials, f)


os.chmod('/root/.kaggle/kaggle.json', 0o600)

print("Kaggle API key configured successfully!")

Kaggle API key configured successfully!


In [2]:
# Download CheXpert-Small
!kaggle datasets download -d ashery/chexpert
!unzip -q chexpert.zip -d /content/chexpert_data
print("CheXpert downloaded and unzipped.")

# Download Multi-Class (Pneumonia, COVID, TB)
!kaggle datasets download -d jtiptj/chest-xray-pneumoniacovid19tuberculosis
!unzip -q chest-xray-pneumoniacovid19tuberculosis.zip -d /content/rural_tb_data
print("Rural TB/Pneumonia dataset downloaded and unzipped.")

Dataset URL: https://www.kaggle.com/datasets/ashery/chexpert
License(s): CC0-1.0
100% 10.7G/10.7G [09:06<00:00, 21.0MB/s]

CheXpert downloaded and unzipped.
Dataset URL: https://www.kaggle.com/datasets/jtiptj/chest-xray-pneumoniacovid19tuberculosis
License(s): other
100% 1.74G/1.74G [01:33<00:00, 20.0MB/s]

Rural TB/Pneumonia dataset downloaded and unzipped.


In [3]:
!pip install -q onnx onnxruntime torch torchvision pandas pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 33.3 MB/s eta 0:00:00


In [11]:
import os
import torch
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import torch.nn as nn

TARGET_CLASSES = ['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Pleural Effusion']

class CheXpertDataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None):
        self.df = pd.read_csv(csv_file)
        self.df[TARGET_CLASSES] = self.df[TARGET_CLASSES].fillna(0)
        self.df[TARGET_CLASSES] = self.df[TARGET_CLASSES].replace(-1, 1)
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        try:
            # Attempt to construct the image path and open it
            img_path = os.path.join(self.root_dir, self.df.iloc[idx]['Path'])
            image = Image.open(img_path).convert('RGB')

            labels = self.df.iloc[idx][TARGET_CLASSES].values.astype('float32')

            if self.transform:
                image = self.transform(image)

            return image, torch.tensor(labels)

        except FileNotFoundError:
            #THE FIX: If the image is missing from Kaggle, quietly skip to the next one!
            return self.__getitem__((idx + 1) % len(self.df))

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("Dataset class defined successfully!")

Dataset class defined successfully!


In [13]:
# Setup Paths (Verify these match where Colab unzipped your files)
CHEXPERT_ROOT = '/content/chexpert_data'
CHEXPERT_CSV = f'{CHEXPERT_ROOT}/train.csv'

# Initialize Dataset and DataLoader
train_dataset = CheXpertDataset(csv_file=CHEXPERT_CSV, root_dir=CHEXPERT_ROOT, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2)

print(f"Loaded {len(train_dataset)} X-ray images for training.")

# Initialize the lightweight MobileNetV2 model
model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)

# Modify the final layer (classifier) for our 5 specific diseases
num_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_features, len(TARGET_CLASSES))

# Move the model to the T4 GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print(f"MobileNetV2 successfully loaded onto: {device}")

Loaded 223414 X-ray images for training.
MobileNetV2 successfully loaded onto: cuda


In [10]:
import torch.optim as optim

# 1. Multi-label Loss Function
criterion = nn.BCEWithLogitsLoss()

# 2. Optimizer (Adam is fast and standard for transfer learning)
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# 3. Training Loop Setup (2 Epochs is plenty for base feature extraction)
EPOCHS = 2

print("Starting Phase 1 Training on CheXpert...")
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0

    # Iterate through the batches of 64 images
    for i, (images, labels) in enumerate(train_loader):
        # Move inputs and labels to the T4 GPU
        images, labels = images.to(device), labels.to(device)

        # Zero the gradients to prevent accumulation from previous steps
        optimizer.zero_grad()

        # Forward pass: Push images through MobileNetV2
        outputs = model(images)

        # Calculate how far off the predictions were from the truth
        loss = criterion(outputs, labels)

        # Backward pass: Calculate gradients
        loss.backward()

        # Optimize: Update the model weights
        optimizer.step()

        running_loss += loss.item()

        # Print progress every 100 batches so you know it hasn't frozen
        if (i + 1) % 100 == 0:
            print(f"Epoch [{epoch+1}/{EPOCHS}], Batch [{i+1}/{len(train_loader)}], Loss: {running_loss/100:.4f}")
            running_loss = 0.0

print("Base Training Completed")

# 4. Save the base model weights
torch.save(model.state_dict(), '/content/mobilenetv2_chexpert_base.pth')
print("Model weights successfully saved to: /content/mobilenetv2_chexpert_base.pth")

Starting Phase 1 Training on CheXpert...


RecursionError: Caught RecursionError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/tmp/ipykernel_1211/3199584077.py", line 27, in __getitem__
    image = Image.open(img_path).convert('RGB')
            ~~~~~~~~~~^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/PIL/Image.py", line 3513, in open
    fp = builtins.open(filename, "rb")
FileNotFoundError: [Errno 2] No such file or directory: '/content/chexpert_data/CheXpert-v1.0-small/train/patient32986/study1/view1_frontal.jpg'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/tmp/ipykernel_1211/3199584077.py", line 27, in __getitem__
    image = Image.open(img_path).convert('RGB')
            ~~~~~~~~~~^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/PIL/Image.py", line 3513, in open
    fp = builtins.open(filename, "rb")
FileNotFoundError: [Errno 2] No such file or directory: '/content/chexpert_data/CheXpert-v1.0-sm

In [14]:
CHEXPERT_ROOT = '/content/chexpert_data'
CHEXPERT_CSV = f'{CHEXPERT_ROOT}/train.csv'

print("Scanning hard drive for corrupted/missing Kaggle files...")
df = pd.read_csv(CHEXPERT_CSV)

# Check which files actually exist on the disk
valid_rows = df['Path'].apply(lambda x: os.path.exists(os.path.join(CHEXPERT_ROOT, x)))

# Filter the dataframe to keep only the valid files
clean_df = df[valid_rows].reset_index(drop=True)

# Save the new, clean CSV
CLEAN_CSV_PATH = '/content/clean_chexpert.csv'
clean_df.to_csv(CLEAN_CSV_PATH, index=False)

print(f"Original Dataset Size: {len(df)}")
print(f"Cleaned Dataset Size:  {len(clean_df)}")
print(f"Deleted {len(df) - len(clean_df)} missing records. Data cleansed")

Scanning hard drive for corrupted/missing Kaggle files...
Original Dataset Size: 223414
Cleaned Dataset Size:  0
Deleted 223414 missing records. Data cleansed


In [ ]:
#skipping the above training and jumping to another dataset

In [16]:
import os
import glob
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

# 1. Auto-detect the exact unzipped training folder
train_dirs = glob.glob('/content/rural_tb_data/**/train', recursive=True)
if not train_dirs:
    raise Exception("Could not find the 'train' folder. Please re-run the Phase 2 download cell!")
TRAIN_DIR = train_dirs[0]

# 2. Image Transformations (Resize for MobileNetV2)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# 3. PyTorch ImageFolder (Automatically reads folders as classes, no CSV needed!)
train_dataset = datasets.ImageFolder(root=TRAIN_DIR, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)

print(f"Found Classes: {train_dataset.classes}")
print(f"Loaded {len(train_dataset)} perfect X-ray images for training.")

# 4. Setup MobileNetV2 Architecture
model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
num_features = model.classifier[1].in_features

# Change output layer to match the number of diseases found (Usually 4)
model.classifier[1] = nn.Linear(num_features, len(train_dataset.classes))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# 5. Loss & Optimizer
# We switch to CrossEntropyLoss because these classes are mutually exclusive
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# 6. Fast Training Loop (5 Epochs will take ~10-15 minutes)
EPOCHS = 5

print("\nStarting Hyper-Specialized Rural Diagnostics Training...")
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        # Print progress every 50 batches
        if (i + 1) % 50 == 0:
            print(f"Epoch [{epoch+1}/{EPOCHS}], Batch [{i+1}/{len(train_loader)}], Loss: {running_loss/50:.4f}")
            running_loss = 0.0

print("Rural Diagnostic Training Complete!")
torch.save(model.state_dict(), '/content/swasthya_vision_model.pth')
print("Model saved to: /content/swasthya_vision_model.pth")

Found Classes: ['COVID19', 'NORMAL', 'PNEUMONIA', 'TURBERCULOSIS']
Loaded 6326 perfect X-ray images for training.

Starting Hyper-Specialized Rural Diagnostics Training...
Epoch [1/5], Batch [50/198], Loss: 0.8276
Epoch [1/5], Batch [100/198], Loss: 0.2956
Epoch [1/5], Batch [150/198], Loss: 0.1935
Epoch [2/5], Batch [50/198], Loss: 0.0938
Epoch [2/5], Batch [100/198], Loss: 0.0753
Epoch [2/5], Batch [150/198], Loss: 0.0766
Epoch [3/5], Batch [50/198], Loss: 0.0485
Epoch [3/5], Batch [100/198], Loss: 0.0380
Epoch [3/5], Batch [150/198], Loss: 0.0533
Epoch [4/5], Batch [50/198], Loss: 0.0202
Epoch [4/5], Batch [100/198], Loss: 0.0188
Epoch [4/5], Batch [150/198], Loss: 0.0271
Epoch [5/5], Batch [50/198], Loss: 0.0129
Epoch [5/5], Batch [100/198], Loss: 0.0104
Epoch [5/5], Batch [150/198], Loss: 0.0212
Rural Diagnostic Training Complete!
Model saved to: /content/swasthya_vision_model.pth


In [17]:
test_dirs = glob.glob('/content/rural_tb_data/**/test', recursive=True)
TEST_DIR = test_dirs[0]

# 2. Same Transforms (NO data augmentation for testing)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

test_dataset = datasets.ImageFolder(root=TEST_DIR, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"Testing model on {len(test_dataset)} unseen X-rays")

# 3. Validation Loop
model.eval() # Turn off dropout/training modes
correct = 0
total = 0

with torch.no_grad(): # Don't track gradients (saves memory & speeds up)
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"Real-World Test Accuracy: {accuracy:.2f}%\n")

Testing model on 771 unseen X-rays
Real-World Test Accuracy: 86.64%



In [20]:
#Exporting model to lightweight ONNX format
!pip install onnxscript
# Create a dummy input tensor that matches the shape of a single X-ray
dummy_input = torch.randn(1, 3, 224, 224).to(device)

onnx_file_path = "/content/swasthya_vision_base.onnx"

torch.onnx.export(
    model,
    dummy_input,
    onnx_file_path,
    export_params=True,
    opset_version=11,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}
)

print(f"SUCCESS! Model exported to: {onnx_file_path}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 754.2/754.2 kB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.8/185.8 kB 19.8 MB/s eta 0:00:00


/tmp/ipykernel_1211/225524466.py:8: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0913 18:44:34.752000 1211 torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 11 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `MobileNetV2([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `MobileNetV2([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/usr/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅


Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/onnxscript/version_converter/__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
        func=_partial_convert_version, model=model
    )
  File "/usr/local/lib/python3.13/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
  File "/usr/local/lib/python3.13/dist-packages/onnxscript/version_converter/__init__.py", line 132, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        proto, target_version=self.target_version
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/onnx/version_converter.py", line 39, in convert_version
    converted_model_str = C.convert_version(model_str, target_version)
RuntimeError: /project/onnx/version_converter/adapters/axes_input_to_attribute.h:56: 

[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
SUCCESS! Model exported to: /content/swasthya_vision_base.onnx
